In [1]:
from regime_detector import RegimeDetector
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import display

In [6]:

def optimize_ema_span(
    detector_base: RegimeDetector,
    *,
    start_date: str,
    end_date: str,
    spans: list[int] = None,
    # scoring weights (higher score is better)
    w_sharpe: float = 1.0,
    w_flip: float = 0.5,
    w_lag: float = 0.5,
    # optional: evaluate in portfolio
    equity_curves: list[dict] | None = None,
    weights_by_regime: dict | None = None,
    initial_equity: float = 1_000_000.0,
    normalize_weights: bool = False,
    missing_data_mode: str = "drop",
) -> pd.DataFrame:
    """
    Grid-search ema_span for RegimeDetector.

    If equity_curves + weights_by_regime are provided, also runs run_portfolio_regime_test()
    and incorporates Sharpe in the score.

    Returns a DataFrame sorted by score desc.
    """
    if spans is None:
        spans = [3, 5, 7, 10, 14, 20, 30, 45, 60]

    # Build an unsmoothed baseline regime sequence (same params, but ema_span=1 ~ minimal smoothing)
    baseline = RegimeDetector(
        vix_high_pct=detector_base.vix_high_pct,
        spread_wide_pct=detector_base.spread_wide_pct,
        lookback=detector_base.lookback,
        credit_mode=detector_base.credit_mode,
        dominance_window=detector_base.dominance_window,
        shift_regime_by_one_day=detector_base.shift_regime_by_one_day,
        ema_span=1,
        ema_min_periods=1,
    )
    base_df = baseline.build_regimes(start_date=start_date, end_date=end_date).dropna()
    base_labels = base_df["RegimeLabel"].astype(str)

    # Optional shift to match your trading alignment
    if detector_base.shift_regime_by_one_day:
        base_labels = base_labels.shift(1).dropna()

    base_labels = base_labels.loc[base_labels.index >= base_df.index.min()]

    def flip_rate(labels: pd.Series) -> float:
        x = labels.astype(str)
        return float((x != x.shift(1)).mean())

    def avg_run_len(labels: pd.Series) -> float:
        x = labels.astype(str)
        change = (x != x.shift(1)).fillna(True)
        run_id = change.cumsum()
        run_lengths = x.groupby(run_id).size()
        return float(run_lengths.mean()) if len(run_lengths) else np.nan

    def transition_lag(smoothed: pd.Series, raw: pd.Series) -> float:
        """
        Proxy lag: fraction of days where raw transitioned but smoothed didn't match raw yet.
        (Lower is better.)
        """
        raw = raw.astype(str).reindex(smoothed.index).dropna()
        sm = smoothed.astype(str).reindex(raw.index).dropna()
        raw_change = (raw != raw.shift(1))
        # days right after a raw change where sm != raw
        lag_days = (raw_change & (sm != raw)).sum()
        denom = max(int(raw_change.sum()), 1)
        return float(lag_days / denom)

    rows = []
    for span in spans:
        det = RegimeDetector(
            vix_high_pct=detector_base.vix_high_pct,
            spread_wide_pct=detector_base.spread_wide_pct,
            lookback=detector_base.lookback,
            credit_mode=detector_base.credit_mode,
            dominance_window=detector_base.dominance_window,
            shift_regime_by_one_day=detector_base.shift_regime_by_one_day,
            ema_span=int(span),
            ema_min_periods=int(span),
        )
        df = det.build_regimes(start_date=start_date, end_date=end_date).dropna()
        labels = df["RegimeLabel"].astype(str)

        if det.shift_regime_by_one_day:
            labels = labels.shift(1).dropna()

        # Align with baseline
        common_idx = labels.index.intersection(base_labels.index)
        labels_c = labels.reindex(common_idx).dropna()
        base_c = base_labels.reindex(common_idx).dropna()

        fr = flip_rate(labels_c)
        arl = avg_run_len(labels_c)
        lag = transition_lag(labels_c, base_c)

        sharpe = np.nan
        if equity_curves is not None and weights_by_regime is not None:
            # Run the portfolio backtest using YOUR pipeline (it will rebuild regimes internally),
            # so we temporarily set ema_span by monkey-patching build_regimes_from_yfinance logic
            # Instead: simplest is to use detector’s regimes directly in a variant of run_portfolio_regime_test.
            # If you want that integrated, say so and I’ll provide a drop-in run_portfolio_regime_test_ema().
            pass

        # Score: higher Sharpe good; lower flip good; lower lag good
        # Convert flip/lag to "goodness" by negating
        score = 0.0
        if np.isfinite(sharpe):
            score += w_sharpe * float(sharpe)
        score += (-w_flip) * float(fr)
        score += (-w_lag) * float(lag)

        rows.append({
            "ema_span": int(span),
            "flip_rate": fr,
            "avg_run_len": arl,
            "transition_lag": lag,
            "portfolio_sharpe": sharpe,
            "score": score,
        })

    out = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return out

In [7]:
det0 = RegimeDetector(
    vix_high_pct=0.70,
    spread_wide_pct=0.70,
    lookback=252,
    credit_mode="ratio",
    ema_span=10,
    shift_regime_by_one_day=True,
)

results = optimize_ema_span(
    det0,
    start_date="2007-01-01",
    end_date=str(pd.Timestamp.today().date()),
    spans=[3,5,7,10,14,20,30,45,60],
    w_flip=0.7,
    w_lag=0.3,
)
results

,ema_span,flip_rate,avg_run_len,transition_lag,portfolio_sharpe,score
0,45,0.012348,80.981818,0.582645,NaN,-0.183437
1,60,0.010137,98.644444,0.588358,NaN,-0.183603
2,3,0.061833,16.172662,0.477273,NaN,-0.186465
3,30,0.015440,64.768116,0.599174,NaN,-0.190560
4,20,0.019647,50.897727,0.609504,NaN,-0.196604
5,14,0.024972,40.044643,0.609504,NaN,-0.200332
6,10,0.030742,32.528986,0.607438,NaN,-0.203751
7,5,0.046284,21.605769,0.584711,NaN,-0.207812
8,7,0.040071,24.955556,0.607438,NaN,-0.210281
